# Employee voice and reported procedural clarity

This notebook examines published aggregate results from the [European Working Conditions Survey 2024, second edition](https://doi.org/10.5255/UKDA-SN-9511-2). The results cover 6,548 eligible employees across the EU27.

Select **Runtime → Run all**. The notebook retrieves the [public research files](https://github.com/evidenceworks/employee-voice-procedures), checks numerical consistency and downloads `employee-voice-results.zip` with three result tables and a verification record.

The checks use published aggregates. Full reestimation from the survey records is documented in `code/reproduce.py` and requires authorized access to the microdata.


## Load the public repository

In [ ]:
from pathlib import Path
from importlib.util import find_spec
import subprocess
import sys
import tempfile

source_url = "https://github.com/evidenceworks/employee-voice-procedures.git"
workspace = Path(tempfile.mkdtemp(prefix="employee-voice-"))
repository = workspace / "repository"
subprocess.run(["git", "clone", "--quiet", "--depth=1", source_url, str(repository)], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(repository), "rev-parse", "HEAD"], text=True
).strip()

# The published numerical checks require only NumPy and SciPy.
needed = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.13,<2"}
missing = [spec for module, spec in needed.items() if find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import numpy
import scipy

print("Public research files loaded.")
print(f"Repository revision: {commit[:12]}")
print(f"Numerical libraries: NumPy {numpy.__version__}, SciPy {scipy.__version__}")

## Check the released estimates

In [ ]:
import json


def run_checked(arguments):
    process = subprocess.run(
        [sys.executable, *arguments], cwd=repository, text=True, capture_output=True
    )
    if process.returncode != 0:
        raise RuntimeError(process.stdout + "\n" + process.stderr)
    return process


integrity = run_checked(["code/verify_scientific_files.py"])
assert "PASS: 20 scientific files" in integrity.stdout

aggregate_check = json.loads(run_checked([
    "code/check_aggregates.py", "--results", "results"
]).stdout)
assert aggregate_check["status"] == "PASS"
assert aggregate_check["empirical_table_rows"] == 350
assert aggregate_check["inferential_records"] == 277

release_tests = run_checked(["-m", "unittest", "discover", "-s", "tests", "-v"])
assert "Ran 2 tests" in release_tests.stderr
assert "OK" in release_tests.stderr

print("Scientific source files: 20 verified")
print("Published aggregate rows: 350")
print("Inferential records checked: 277")
print("Response partitions, confidence intervals and Holm adjustments: verified")
print("Release tests: 2 passed")

## Prepare the results

In [ ]:
import csv
import hashlib
import platform
import zipfile
from scipy.stats import t as student_t


def load_rows(filename):
    with (repository / "results" / filename).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def write_rows(path, fields, rows):
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def formatted_probability(record):
    estimate = float(record["estimate_pp"])
    standard_error = float(record["se_pp"])
    return f"{2 * student_t.sf(abs(estimate / standard_error), 2455):.10g}"


export = workspace / "results"
export.mkdir()

# A compact four-group summary of the 12 published response cells.
response_rows = load_rows("T1_Procedure_by_voice.csv")
profile_output = []
for group, label in [
    ("neither", "Neither"),
    ("representation_only", "Representation only"),
    ("meetings_only", "Meetings only"),
    ("both", "Both"),
]:
    cells = {row["outcome"]: row for row in response_rows if row["group"] == group}
    assert set(cells) == {"1", "2", "3"}
    n = int(cells["1"]["group_n"])
    assert all(int(row["group_n"]) == n for row in cells.values())
    profile_output.append({
        "voice_configuration": label,
        "employees_n": n,
        "clear_procedure_pct": f"{float(cells['1']['weighted_pct']):.4f}",
        "no_clear_procedure_pct": f"{float(cells['2']['weighted_pct']):.4f}",
        "does_not_know_pct": f"{float(cells['3']['weighted_pct']):.4f}",
    })
assert sum(row["employees_n"] for row in profile_output) == 6548
write_rows(export / "voice_profiles.csv", list(profile_output[0]), profile_output)

# Released adjusted contrasts for the three procedural responses.
t7_rows = load_rows("T7_Model_contrasts.csv")
contrasts = {(row["model"], row["contrast"]): row for row in t7_rows}
baseline = {(row["outcome"], row["contrast"]): row for row in load_rows("BASELINE_Q57.csv")}
models = [
    ("procedure_yes", "A_procedure_yes", "Reports a clear procedure"),
    ("procedure_no", "A_procedure_no", "Reports no clear procedure"),
    ("procedure_unknown", "A_procedure_unknown", "Does not know"),
]
contrast_names = [
    "representation_only - neither",
    "meetings_only - neither",
    "both - neither",
    "both - representation_only",
    "both - meetings_only",
    "meetings_only - representation_only",
    "additive_interaction",
]
contrast_output = []
for outcome, model, outcome_label in models:
    for name in contrast_names:
        derived = name == "both - neither"
        row = baseline[(outcome, name)] if derived else contrasts[(model, name)]
        contrast_output.append({
            "response": outcome_label,
            "contrast": name.replace("_", " "),
            "employees_n": int(float(row["n"])),
            "estimate_pp": f"{float(row['estimate_pp']):.8f}",
            "se_pp": f"{float(row['se_pp']):.8f}",
            "ci95_lower_pp": f"{float(row['lo95_pp']):.8f}",
            "ci95_upper_pp": f"{float(row['hi95_pp']):.8f}",
            "p_value": formatted_probability(row),
            "holm_p_primary": row.get("holm_p_primary", ""),
            "source": "published derived contrast" if derived else "published model contrast",
        })
assert len(contrast_output) == 21
write_rows(export / "adjusted_contrasts.csv", list(contrast_output[0]), contrast_output)

# Both focused restrictions are shown beside the full-sample interaction.
sensitivity_output = []
for label, model, outcome in [
    ("Main analytic sample", "A_procedure_yes", "Reports a clear procedure"),
    ("Main analytic sample", "A_procedure_unknown", "Does not know"),
    ("High-intensity computer-mediated functions", "A_large", "Reports a clear procedure"),
    ("Definite yes/no responses", "A_yesno", "Reports a clear procedure"),
]:
    row = contrasts[(model, "additive_interaction")]
    sensitivity_output.append({
        "sample": label,
        "response": outcome,
        "employees_n": int(float(row["n"])),
        "additive_interaction_pp": f"{float(row['estimate_pp']):.8f}",
        "ci95_lower_pp": f"{float(row['lo95_pp']):.8f}",
        "ci95_upper_pp": f"{float(row['hi95_pp']):.8f}",
        "p_value": formatted_probability(row),
    })
assert [row["employees_n"] for row in sensitivity_output] == [6548, 6548, 3148, 5581]
write_rows(export / "sensitivity_results.csv", list(sensitivity_output[0]), sensitivity_output)

# Record the exact public revision and the numerical checks that were run.
exported_csvs = sorted(export.glob("*.csv"))
verification = {
    "study": "European Working Conditions Survey 2024, second edition; UK Data Service 9511",
    "repository": "https://github.com/evidenceworks/employee-voice-procedures",
    "commit": commit,
    "scope": "Checks and summaries of published aggregate results",
    "environment": {
        "python": platform.python_version(),
        "numpy": numpy.__version__,
        "scipy": scipy.__version__,
    },
    "checks": {
        "scientific_files_verified": 20,
        "aggregate_rows_checked": aggregate_check["empirical_table_rows"],
        "inferential_records_checked": aggregate_check["inferential_records"],
        "release_tests_passed": 2,
        "confidence_interval_reconstruction": "verified",
        "response_partition": aggregate_check["response_partition"],
        "holm_adjustments": aggregate_check["three_contrast_Holm"],
        "country_denominator": aggregate_check["country_denominator"],
    },
    "exported_csv_sha256": {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in exported_csvs
    },
}
(export / "verification.json").write_text(
    json.dumps(verification, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

(export / "README.txt").write_text(
    "Employee voice and reported procedural clarity\n"
    "European Working Conditions Survey 2024 (second edition), UK Data Service study 9511\n\n"
    "Source: https://github.com/evidenceworks/employee-voice-procedures\n"
    f"Git revision: {commit}\n\n"
    "voice_profiles.csv: Unweighted group counts and survey-weighted response percentages.\n"
    "adjusted_contrasts.csv: Adjusted probability differences and 95% confidence intervals,\n"
    "including three published both-versus-neither derived contrasts.\n"
    "sensitivity_results.csv: Main additive interactions and two focused restrictions.\n"
    "verification.json: Numerical checks, library versions and CSV checksums.\n\n"
    "Differences and intervals are in percentage points. The response 'does not know'\n"
    "is distinct from reporting that no clear procedure exists. Results are associations\n"
    "among reported responses, not verified organizational procedures.\n\n"
    "These exports summarize publicly released aggregates. Reestimation from the survey\n"
    "records is described in code/reproduce.py and requires authorized UKDS access.\n",
    encoding="utf-8"
)

archive = Path.cwd() / "employee-voice-results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as package:
    for path in sorted(export.iterdir()):
        package.write(path, arcname=path.name)

with zipfile.ZipFile(archive) as package:
    assert package.testzip() is None
    assert set(package.namelist()) == {
        "voice_profiles.csv", "adjusted_contrasts.csv", "sensitivity_results.csv",
        "verification.json", "README.txt"
    }
print(f"Created {archive.name} ({archive.stat().st_size:,} bytes)")

## View and download

In [ ]:
from IPython.display import Markdown, display

display(Markdown("### Procedural responses by voice configuration"))
display(Markdown(
    "| Configuration | Employees | Clear (%) | No clear procedure (%) | Does not know (%) |\n"
    "|:--|--:|--:|--:|--:|\n" + "\n".join(
        f"| {r['voice_configuration']} | {r['employees_n']:,} | "
        f"{float(r['clear_procedure_pct']):.1f} | "
        f"{float(r['no_clear_procedure_pct']):.1f} | "
        f"{float(r['does_not_know_pct']):.1f} |"
        for r in profile_output
    )
))
display(Markdown("### Additive interactions and sensitivity"))
display(Markdown(
    "| Sample | Response | Difference (pp) | 95% CI (pp) |\n"
    "|:--|:--|--:|:--|\n" + "\n".join(
        f"| {r['sample']} | {r['response']} | "
        f"{float(r['additive_interaction_pp']):+.2f} | "
        f"[{float(r['ci95_lower_pp']):+.2f}, {float(r['ci95_upper_pp']):+.2f}] |"
        for r in sensitivity_output
    )
))

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print(f"Results saved to {archive}")